# Sales File Validator — North Region

Checks the "Base File Updation" workbooks returned by sales against the
blank files they were originally sent, and reports anything that looks
wrong — **before** the files get combined into the company-wide report.

**What "wrong" means here, in plain terms:**
- A label at the top of a column got typed over by accident
- A row is half-filled in (a part number with no customer code, or vice versa)
- A number that should never change (this year's official budget, a customer's
  name, a fixed formula) came back different
- A formula that should calculate automatically got overwritten with a
  typed-in number, so it will quietly go stale later
- A calculation's result doesn't actually match what its inputs say it should

**What this notebook does NOT do:** it doesn't fix anything automatically.
It only tells you, in plain language, exactly what changed, where, and how
serious it is — so a person can decide what to do next.

**To run this each cycle:**
1. Put the blank files you sent out in the `sent_files/` folder.
2. Put the files sales returned in the `received_files/` folder.
3. Check the 3 settings under "Configuration" below match this cycle
   (which month is the active forecast month, etc.) — this is the *only*
   thing that should ever need changing month to month.
4. Run all cells. Read the plain-English summary at the very end first —
   it tells you immediately whether everything's fine or something needs
   a look, and points you to the full detail in the Excel report it saves.


In [1]:
import os
import re
import io
import glob
from collections import defaultdict, Counter
from datetime import datetime

import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, PatternFill, Alignment

try:
    import msoffcrypto
except ImportError:
    msoffcrypto = None


## 1. Configuration

Everything that's specific to *this cycle* lives here. Nothing below this
section should need editing month to month — only these settings do.


In [2]:
# --- folders ---------------------------------------------------------------
from pathlib import Path

SENT_DIR = Path("input_files/sent_files")  # the blank files you sent out to sales
RECEIVED_DIR = Path("input_files/received_files") # the files sales sent back
REPORT_DIR = "validation_reports" # where this notebook saves its findings

print("SENT FILES:")
for file in SENT_DIR.iterdir():
    print(" -", file.name)

print("\nRECEIVED FILES:")
for file in RECEIVED_DIR.iterdir():
    print(" -", file.name)





# --- password used ONLY if a file turns out to be password protected -------
FILE_PASSWORD = "raja"

# --- what sales is allowed to touch THIS cycle ------------------------------
# Change these three lines each cycle; everything else in the notebook reads
# from them, so nothing else needs to change.
CONFIG = {
    "active_forecast_month": "Nov",              # the month sales is filling in right now
    "catchup_forecast_months": ["Sep", "Oct"],    # the 1-2 months just before it, allowed light corrections
    "active_incoming_month": "Oct",               # the "Incoming" (shipment) column currently allowed to change
    "thresholds": {
        "pat_suspicious_pct": 50,   # % of rows changed in one allowed column before we flag it as unusual
        "cal_tolerance": 0.01,      # allowed rounding gap when checking Qty x Price = Value
    },
}

RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
REPORT_FILE = os.path.join(REPORT_DIR, f"validation_report_{RUN_TIMESTAMP}.xlsx")

print("Active forecast month:", CONFIG["active_forecast_month"])
print("Catch-up months:", CONFIG["catchup_forecast_months"])
print("Active Incoming month:", CONFIG["active_incoming_month"])


SENT FILES:
 - Kuldeep-Base File Updation North Region -Sep-26.xlsx
 - Arpit-Base File Updation North Region -Sep-26.xlsx
 - Abhishekh-Base File Updation North Region -Sep-26 working File - Copy (6).xlsx
 - Mahesh-Base File Updation North Region -Sep-26 working File - Copy (5).xlsx
 - Suraj-Base File Updation North Region -Sep-26 working File - Copy (3).xlsx
 - Harsh-Base File Updation North Region -Sep-26 working File - Copy (4).xlsx

RECEIVED FILES:
 - Kuldeep-Base File Updation North Region -Sep-26.xlsx
 - Arpit-Base File Updation North Region -Sep-26.xlsx
 - Abhishekh-Base File Updation North Region -Sep-26 working File - Copy (6).xlsx
 - Mahesh-Base File Updation North Region -Sep-26 working File - Copy (5).xlsx
 - Suraj-Base File Updation North Region -Sep-26 working File.xlsx
 - Copy of Harsh-Base File Updation North Region -Sep-26 working File - Copy (4).xlsx
Active forecast month: Nov
Catch-up months: ['Sep', 'Oct']
Active Incoming month: Oct


## 2. What sales is / isn't allowed to touch, and how columns are recognized

This section defines the rule engine's vocabulary — it reads every column
header and works out what *kind* of column it is (a master-data field, this
cycle's forecast, a fixed budget number, a formula, etc.), then decides
whether sales is allowed to have changed it, based on the `CONFIG` above.

You shouldn't need to edit this section — it works from the header *text*,
not a hard-coded column list, so it keeps working even if columns shift
position or a sheet has slightly different spacing in its headers.


In [3]:
SHEET_ALIASES = {
    "INR Sheet": ["INR Sheet", "INR Sheet "],
    "USD Sheet": ["USD Sheet", "USD Sheet "],
    "Direct Sheet": ["Direct Sheet", "Direct Sheet "],
}
CANONICAL_SHEETS = ["INR Sheet", "USD Sheet", "Direct Sheet"]
HEADER_ROW = 2
DATA_START_ROW = 3

MASTER_COLUMNS = {
    "currency", "zone", "region", "customer code", "customer name", "updated sales",
    "updates sale person name", "sales csr", "principal", "make", "pm names", "pm mso",
    "pm csr", "part number", "segment", "sub segment", "end customer name",
    "end customer segment", "end customer sub-segment",
}
MONTH_ORDER = ["Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec","Jan","Feb","Mar"]

# Columns in these categories are ALWAYS a formula (Qty x Price, or a roll-up
# sum of several months) — never something a person types directly. They are
# never checked for "did this change", only for "does it still calculate
# correctly" (see the CAL and formula-integrity checks further down).
DERIVED_CATEGORIES = {"budget_value", "forecast_value", "summary_formula"}


def classify_header(header):
    """
    Works out what kind of column a header represents, from its text alone.
    Returns {"category": ..., "month": "Apr".."Mar" or None}.
    """
    if header is None:
        return {"category": "unclassified", "month": None}
    h = header.strip()
    hl = h.lower()

    if hl in MASTER_COLUMNS:
        return {"category": "master", "month": None}
    if hl == "existing/new":
        return {"category": "existing_new", "month": None}
    if "ep price" in hl or "ep in us" in hl:
        return {"category": "ep_price", "month": None}
    if hl == "remarks":
        return {"category": "remarks", "month": None}
    if "month total inventory" in hl or "month inventory" in hl:
        return {"category": "inventory_qty", "month": None}
    if hl.startswith("all month budget") or "to mar-27" in hl or "to mar 27" in hl:
        return {"category": "summary_formula", "month": None}

    m = re.match(r"^([A-Za-z]+)", h)
    if m:
        month3 = m.group(1)[:3].title()
        if month3 in MONTH_ORDER:
            is_incoming = "incoming" in hl
            is_forecast = bool(re.search(r"\(f\)", hl))
            is_budget = bool(re.search(r"\(b\)", hl)) or bool(re.search(r"\bb\s+value\b", hl))
            is_qty = "qty" in hl
            is_value = "value" in hl
            if is_incoming:
                return {"category": "incoming", "month": month3}
            if is_forecast and is_qty:
                return {"category": "forecast_qty", "month": month3}
            if is_forecast and is_value:
                return {"category": "forecast_value", "month": month3}
            if is_budget and is_qty:
                return {"category": "budget_qty", "month": month3}
            if is_budget and is_value:
                return {"category": "budget_value", "month": month3}

    return {"category": "unclassified", "month": None}


def is_allowed(category, month, config):
    """
    True only for the direct-input categories sales is explicitly permitted
    to edit this cycle. Everything else defaults to PROTECTED — a column has
    to be explicitly allowed here, it is never assumed safe to edit.
    (Derived/formula categories are handled separately, not through this
    function — see DERIVED_CATEGORIES above.)
    """
    if category == "forecast_qty":
        return month == config["active_forecast_month"] or month in config["catchup_forecast_months"]
    if category == "incoming":
        return month == config["active_incoming_month"]
    if category in ("ep_price", "remarks"):
        return True
    return False


## 3. Reading files (including password-protected ones)

Every file is opened the same way whether or not it's locked — if
`msoffcrypto` reports a file as encrypted, it's decrypted on the fly using
`FILE_PASSWORD` above; otherwise it's opened normally.


In [4]:
def open_workbook(path, password=FILE_PASSWORD, data_only=False):
    is_encrypted = False
    if msoffcrypto is not None:
        try:
            with open(path, "rb") as fh:
                is_encrypted = msoffcrypto.OfficeFile(fh).is_encrypted()
        except Exception:
            is_encrypted = False
    if not is_encrypted:
        return openpyxl.load_workbook(path, data_only=data_only)
    decrypted = io.BytesIO()
    with open(path, "rb") as fh:
        office_file = msoffcrypto.OfficeFile(fh)
        office_file.load_key(password=password)
        office_file.decrypt(decrypted)
    decrypted.seek(0)
    return openpyxl.load_workbook(decrypted, data_only=data_only)


def find_sheet(wb, canonical_name):
    for alias in SHEET_ALIASES[canonical_name]:
        if alias in wb.sheetnames:
            return alias
    return None


def clean_header(v):
    if v is None:
        return None
    return " ".join(str(v).replace("\n", " ").split())


def person_name(filepath):
    """The salesperson's name is the text before the first '-' in the filename,
    with a leading 'Copy of' stripped off (some return files get renamed that
    way by whoever forwards them)."""
    return os.path.basename(filepath).split("-")[0].replace("Copy of", "").strip()


## 4. Findings — every issue this notebook detects gets recorded the same way

Each finding is a small record: which rule caught it, how severe it is, who
sent the file, which sheet/record/column it's about, and a plain-English
explanation. Nothing is auto-corrected — the notebook only records and
reports.


In [5]:
FINDINGS = []

def add_finding(rule, severity, person, sheet, **kw):
    f = {"rule": rule, "severity": severity, "person": person, "sheet": sheet}
    f.update(kw)
    FINDINGS.append(f)

# Plain-English name for every rule code, used in the report and the summary.
RULE_DESCRIPTIONS = {
    "INT-002": "File pairing problem (a sent or received file has no match)",
    "STR-001": "A required sheet is missing from the return file",
    "STR-002/003": "A column header has moved, changed, or is missing",
    "STR-004": "The same column header appears more than once",
    "STR-005": "Unexpected sheet, or key columns could not be found",
    "STR-006": "This person's sent file doesn't match the layout everyone else was sent",
    "ROW-004": "The same Customer+Part combination now appears more than once",
    "ROW-005": "An incomplete row (Customer Code or Part Number is missing)",
    "ROW-007": "A record that was sent is missing from the return file",
    "ROW-008": "A record appears that was not in the sent file",
    "PRT": "Data changed in a column that should never change",
    "CAL-001": "A formula's result doesn't match what its inputs say it should be",
    "PAT-001": "An allowed column changed on an unusually large share of rows",
}


## 5. Stage 1 — Find and pair up the sent and received files

Files aren't always returned under the exact same name (e.g. `Copy of
Harsh-...xlsx`), so pairing is done by salesperson name, not exact filename.


In [6]:
sent_files = sorted(glob.glob(os.path.join(SENT_DIR, "*.xlsx")))
recv_files = sorted(glob.glob(os.path.join(RECEIVED_DIR, "*.xlsx")))

sent_by_person = {person_name(f): f for f in sent_files}
recv_by_person = {person_name(f): f for f in recv_files}

for person in sent_by_person:
    if person not in recv_by_person:
        add_finding("INT-002", "CRITICAL", person, "-",
                     message=f"{person} was sent a file but no return file was found.")
for person in recv_by_person:
    if person not in sent_by_person:
        add_finding("INT-002", "CRITICAL", person, "-",
                     message=f"A return file for '{person}' was found but no matching sent file exists.")

PAIRS = {p: (sent_by_person[p], recv_by_person[p]) for p in sent_by_person if p in recv_by_person}

print(f"Sent files found: {len(sent_files)}")
print(f"Received files found: {len(recv_files)}")
print(f"Successfully paired: {len(PAIRS)} -> {list(PAIRS.keys())}")
if len(PAIRS) != len(sent_files) or len(PAIRS) != len(recv_files):
    print("NOTE: not every file could be paired — see INT-002 findings in the final report.")


Sent files found: 6
Received files found: 6
Successfully paired: 6 -> ['Abhishekh', 'Arpit', 'Harsh', 'Kuldeep', 'Mahesh', 'Suraj']


## 6. Stage 1b — Make sure the SENT files all match each other

Before comparing anything to what came back, we check that the files sent out were themselves consistent — same columns, same order, on every sheet, for every salesperson. If one person was accidentally sent an outdated or different template, this catches it here, rather than confusing every check that follows.

In [7]:
for canonical in CANONICAL_SHEETS:
    person_headers = {}
    for person, sent_path in sent_by_person.items():
        wb = open_workbook(sent_path, data_only=False)
        sheet_name = find_sheet(wb, canonical)
        if sheet_name is None:
            continue
        ws = wb[sheet_name]
        headers = tuple(clean_header(ws.cell(row=HEADER_ROW, column=c).value) for c in range(1, ws.max_column+1))
        person_headers[person] = headers

    if len(person_headers) < 2:
        continue  # nothing to compare this sheet against

    # the layout used by the most people is treated as the expected baseline
    counts = Counter(person_headers.values())
    baseline, _ = counts.most_common(1)[0]

    for person, headers in person_headers.items():
        if headers == baseline:
            continue
        max_len = max(len(headers), len(baseline))
        diffs = []
        for i in range(max_len):
            b = baseline[i] if i < len(baseline) else None
            h = headers[i] if i < len(headers) else None
            if b != h:
                diffs.append(f"column {get_column_letter(i+1)}: expected '{b}', found '{h}'")
        add_finding("STR-006", "CRITICAL", person, canonical,
                     message="This person was sent a file with a different layout than everyone else. "
                             + "; ".join(diffs[:5]) + (" ..." if len(diffs) > 5 else ""))

sent_consistency_issues = [f for f in FINDINGS if f["rule"] == "STR-006"]
if sent_consistency_issues:
    print(f"NOTE: {len(sent_consistency_issues)} sent-file layout mismatch(es) found -- see STR-006 in the report.")
    print("The files sent out to sales were not all identical before this cycle even started.")
else:
    print("All sent files use the same column layout as each other -- good starting point.")


All sent files use the same column layout as each other -- good starting point.


## 7. Stage 2 — The main validation engine

For every paired file, every applicable sheet:

1. **Structure check** — every header must be exactly where and what it was
   in the sent file (catches a corrupted or moved column label).
2. **Row identity** — every record is matched between sent and received by
   **Customer Code + Part Number**, never by row number, because a single
   inserted or deleted row would otherwise silently misalign every row
   below it.
3. **Protected-data check** — for every matched record, every column
   outside this cycle's allowed list must be byte-for-byte identical to
   what was sent.
4. **Formula-integrity check** — every formula-driven column (Value
   columns, and the 4 roll-up totals) must still actually be a formula, not
   a number someone pasted over it.
5. **Calculation check** — every formula-driven column's *result* must
   still match what its inputs say it should compute, catching broken
   references and formulas that silently return the wrong number.
6. **Pattern check** — even for columns sales is allowed to edit, an
   unusually high share of rows changing in one column is flagged for a
   second look.


In [8]:
TOL = CONFIG["thresholds"]["cal_tolerance"]

for person, (sent_path, recv_path) in PAIRS.items():
    wb_sent_f = open_workbook(sent_path, data_only=False)
    wb_recv_f = open_workbook(recv_path, data_only=False)
    wb_sent_v = open_workbook(sent_path, data_only=True)
    wb_recv_v = open_workbook(recv_path, data_only=True)

    for canonical in CANONICAL_SHEETS:
        sent_sheet = find_sheet(wb_sent_f, canonical)
        recv_sheet = find_sheet(wb_recv_f, canonical)

        if sent_sheet is None and recv_sheet is None:
            continue  # this person never had this sheet -> not applicable
        if sent_sheet is None and recv_sheet is not None:
            add_finding("STR-005", "CRITICAL", person, canonical,
                         message="This sheet appears in the return file but was never sent.")
            continue
        if sent_sheet is not None and recv_sheet is None:
            add_finding("STR-001", "CRITICAL", person, canonical,
                         message="This required sheet is missing from the return file.")
            continue

        ws_sf, ws_rf = wb_sent_f[sent_sheet], wb_recv_f[recv_sheet]
        ws_sv, ws_rv = wb_sent_v[sent_sheet], wb_recv_v[recv_sheet]

        sent_headers = [clean_header(ws_sf.cell(row=HEADER_ROW, column=c).value) for c in range(1, ws_sf.max_column+1)]
        recv_headers = [clean_header(ws_rf.cell(row=HEADER_ROW, column=c).value) for c in range(1, ws_rf.max_column+1)]

        # ---- STR-004: duplicate headers in the return file ----
        seen = {}
        for i, h in enumerate(recv_headers, start=1):
            if h is None:
                continue
            if h in seen:
                add_finding("STR-004", "CRITICAL", person, canonical, column=h,
                             message=f"Header '{h}' appears more than once (columns {seen[h]} and {get_column_letter(i)}).")
            else:
                seen[h] = get_column_letter(i)

        # ---- STR-002/003: every header must match the sent file, in the same column ----
        max_len = max(len(sent_headers), len(recv_headers))
        header_break = False
        for i in range(max_len):
            s_h = sent_headers[i] if i < len(sent_headers) else None
            r_h = recv_headers[i] if i < len(recv_headers) else None
            if s_h != r_h:
                header_break = True
                add_finding("STR-002/003", "CRITICAL", person, canonical, column=get_column_letter(i+1),
                             message=f"Column {get_column_letter(i+1)} should be '{s_h}' but is now '{r_h}'.")
        if header_break:
            # Column names can no longer be trusted for this sheet -> stop here for it.
            # (Everything else about this file is still checked normally.)
            continue

        headers = sent_headers
        code_col = next((i+1 for i,h in enumerate(headers) if h and h.lower()=="customer code"), None)
        part_col = next((i+1 for i,h in enumerate(headers) if h and h.lower()=="part number"), None)
        if not code_col or not part_col:
            add_finding("STR-005", "CRITICAL", person, canonical,
                         message="Could not find the Customer Code / Part Number columns.")
            continue

        # ---- ROW: build a Customer Code + Part Number index for both files ----
        def build_index(ws, max_row):
            idx = defaultdict(list)
            orphans = []
            for r in range(DATA_START_ROW, max_row+1):
                code = ws.cell(row=r, column=code_col).value
                part = ws.cell(row=r, column=part_col).value
                if code is None and part is None:
                    continue
                if (code is None) != (part is None):
                    orphans.append((r, code, part))
                idx[(code, part)].append(r)
            return idx, orphans

        idx_sent, _ = build_index(ws_sv, ws_sv.max_row)
        idx_recv, orphans_recv = build_index(ws_rv, ws_rv.max_row)

        for r, code, part in orphans_recv:
            add_finding("ROW-005", "WARNING", person, canonical, row=r, customer_code=code, part_number=part,
                         message=f"Row {r} has only one of Customer Code / Part Number filled in — looks like an incomplete entry.")

        sent_keys, recv_keys = set(idx_sent.keys()), set(idx_recv.keys())

        for k in sent_keys - recv_keys:
            add_finding("ROW-007", "CRITICAL", person, canonical, customer_code=k[0], part_number=k[1],
                         message=f"This record (Customer Code={k[0]}, Part Number={k[1]}) was sent but is missing from the return file.")

        codes_in_sent_file = {k[0] for k in sent_keys if k[0] is not None}
        for k in recv_keys - sent_keys:
            if k[0] is None or k[1] is None:
                continue  # already caught above as an incomplete row
            if k[0] in codes_in_sent_file:
                context = "This customer already exists elsewhere in the file — likely a legitimate new part added."
            else:
                context = "This customer doesn't exist anywhere else in the file either — worth double-checking."
            add_finding("ROW-008", "WARNING", person, canonical, customer_code=k[0], part_number=k[1],
                         message=f"This record (Customer Code={k[0]}, Part Number={k[1]}) appears in the return file but wasn't in the sent file. {context}")

        for k, rows in idx_recv.items():
            if k[0] is None or k[1] is None:
                continue
            if len(rows) > 1 and len(idx_sent.get(k, [])) <= 1:
                add_finding("ROW-004", "WARNING", person, canonical, customer_code=k[0], part_number=k[1],
                             message=f"This record (Customer Code={k[0]}, Part Number={k[1]}) now appears {len(rows)} times — it only appeared once in the sent file.")

        # ---- classify every column, then check protected data + formulas ----
        col_class = {i+1: classify_header(h) for i,h in enumerate(headers) if h}
        ep_col = next((c for c,cls in col_class.items() if cls["category"]=="ep_price"), None)

        matched_keys = sent_keys & recv_keys
        change_counts, total_counts = defaultdict(int), defaultdict(int)

        for key in matched_keys:
            for r_s, r_r in zip(idx_sent[key], idx_recv[key]):
                for c, cls in col_class.items():
                    h = headers[c-1]
                    v_s = ws_sv.cell(row=r_s, column=c).value
                    v_r = ws_rv.cell(row=r_r, column=c).value
                    changed = (v_s != v_r)
                    total_counts[h] += 1
                    if changed:
                        change_counts[h] += 1

                    cat = cls["category"]

                    # ---- Value columns and roll-up totals: never a direct sales
                    #      input, so never checked for "did it change" — only
                    #      "does it still calculate correctly" ----
                    if cat in DERIVED_CATEGORIES:
                        if cat == "summary_formula":
                            hl = h.lower()
                            if "budget" in hl and "qty" in hl: src_cat, divisor = "budget_qty", 1000
                            elif "budget" in hl and "value" in hl: src_cat, divisor = "budget_value", 1000
                            elif "incoming" in hl: src_cat, divisor = "incoming", 1
                            elif "value" in hl: src_cat, divisor = "forecast_value", 1
                            else: src_cat, divisor = None, 1
                            if not src_cat:
                                continue
                            rollup_months = {"Sep","Oct","Nov","Dec","Jan","Feb","Mar"} if src_cat in ("incoming","forecast_value") else set(MONTH_ORDER)
                            total = sum(
                                ws_rv.cell(row=r_r, column=cc).value
                                for cc, cc_cls in col_class.items()
                                if cc_cls["category"] == src_cat and cc_cls["month"] in rollup_months
                                and isinstance(ws_rv.cell(row=r_r, column=cc).value, (int, float))
                            )
                            expected = total / divisor
                            label = "roll-up total"
                        else:
                            qty_cat = "budget_qty" if cat == "budget_value" else "forecast_qty"
                            qty_col_match = next((cc for cc,cc_cls in col_class.items()
                                                   if cc_cls["category"]==qty_cat and cc_cls["month"]==cls["month"]), None)
                            if not qty_col_match or not ep_col:
                                continue
                            q_raw = ws_rv.cell(row=r_r, column=qty_col_match).value
                            p_raw = ws_rv.cell(row=r_r, column=ep_col).value
                            q = q_raw if isinstance(q_raw, (int, float)) else 0
                            p = p_raw if isinstance(p_raw, (int, float)) else 0
                            expected = q * p
                            label = "formula"

                        if not isinstance(v_r, (int, float)) and v_r is not None:
                            add_finding("CAL-001", "CRITICAL", person, canonical,
                                         customer_code=key[0], part_number=key[1], column=h,
                                         message=f"'{h}' (a {label}) contains a non-numeric value ({v_r!r}) where a calculated number was expected.")
                        else:
                            actual = v_r if isinstance(v_r, (int, float)) else 0
                            if abs(expected - actual) > max(TOL, abs(expected) * TOL):
                                add_finding("CAL-001", "CRITICAL", person, canonical,
                                             customer_code=key[0], part_number=key[1], column=h,
                                             message=f"'{h}' = {actual!r}, but its inputs say it should be {expected!r} — the {label} result doesn't match.")
                        continue

                    # ---- Everything else: protected unless explicitly allowed ----
                    allowed = is_allowed(cat, cls["month"], CONFIG)
                    if changed and not allowed:
                        add_finding("PRT", "CRITICAL", person, canonical,
                                     customer_code=key[0], part_number=key[1], column=h,
                                     old_value=v_s, new_value=v_r,
                                     message=f"'{h}' should never change, but went from {v_s!r} to {v_r!r}.")

        # ---- PAT-001: unusually high change volume, even in an allowed column ----
        for h, changed in change_counts.items():
            if total_counts[h] == 0:
                continue
            if classify_header(h)["category"] in DERIVED_CATEGORIES:
                continue  # expected to move whenever its driver column changes
            pct = 100 * changed / total_counts[h]
            if pct >= CONFIG["thresholds"]["pat_suspicious_pct"]:
                add_finding("PAT-001", "WARNING", person, canonical, column=h,
                             message=f"'{h}' changed on {changed}/{total_counts[h]} rows ({pct:.0f}%) — unusually high, worth a second look even though this column is allowed to change.")

print(f"Validation complete. Total findings: {len(FINDINGS)}")


Validation complete. Total findings: 375


## 8. Save the full findings to an Excel report

Every finding, one row per issue, so nothing is summarized away — this is
the file to open when you need the exact record, column, and values behind
any issue mentioned in the summary below.


In [9]:
os.makedirs(REPORT_DIR, exist_ok=True)

HEADER_FILL = PatternFill("solid", fgColor="1F4E78")
HEADER_FONT = Font(bold=True, color="FFFFFF")
CRITICAL_FILL = PatternFill("solid", fgColor="F8CBCB")
WARNING_FILL = PatternFill("solid", fgColor="FCEBA4")

wb_report = openpyxl.Workbook()

# ---- "How To Use This Report" guide sheet (placed first, so it's the
#      first thing anyone sees when opening the file — no other document
#      should ever be needed to understand a finding in this report) ----
wb_report.remove(wb_report.active)
ws0 = wb_report.create_sheet("How To Use This Report", 0)

TITLE_FONT = Font(bold=True, size=14, color="1F4E78")
SECTION_FONT = Font(bold=True, size=12, color="FFFFFF")
SECTION_FILL = PatternFill("solid", fgColor="1F4E78")
SUBHEAD_FONT = Font(bold=True, size=11)
WRAP = Alignment(wrap_text=True, vertical="top")
GOOD_FILL_GUIDE = PatternFill("solid", fgColor="C6EFCE")

def guide_row(ws, r, values, fills=None, fonts=None, wrap_cols=None):
    for i, v in enumerate(values, start=1):
        cell = ws.cell(row=r, column=i, value=v)
        if fills and i in fills:
            cell.fill = fills[i]
        if fonts and i in fonts:
            cell.font = fonts[i]
        if wrap_cols and i in wrap_cols:
            cell.alignment = WRAP
    return r + 1

r = 1
ws0.cell(row=r, column=1, value="Sales File Validator — How To Use This Report").font = TITLE_FONT
r += 2

intro = ("This report checks the files sales returned against the blank files they were sent, "
         "before those files get combined into the company-wide report. Everything you need to "
         "understand any finding below is on this one sheet — you shouldn't need any other document.")
ws0.merge_cells(start_row=r, start_column=1, end_row=r, end_column=6)
ws0.cell(row=r, column=1, value=intro).alignment = WRAP
ws0.row_dimensions[r].height = 40
r += 2

ws0.cell(row=r, column=1, value="What's in this workbook").font = SUBHEAD_FONT
r += 1
tabs = [
    ("Summary", "One line per salesperson — how many Critical / Warning findings, and an overall status."),
    ("Summary By Category", "Same, but broken down by the 6 error categories below — shows at a glance WHAT KIND of problem each person has."),
    ("Findings", "Every single issue found, one row each, with the exact customer, part, column, old value and new value — the full detail."),
]
for name, desc in tabs:
    ws0.cell(row=r, column=1, value=name).font = Font(bold=True)
    ws0.merge_cells(start_row=r, start_column=2, end_row=r, end_column=6)
    ws0.cell(row=r, column=2, value=desc).alignment = WRAP
    r += 1
r += 1

ws0.cell(row=r, column=1, value="Severity levels").font = SUBHEAD_FONT
r += 1
sev_rows = [
    ("CRITICAL", CRITICAL_FILL, "Real data corruption or lost/incorrect information. Always needs to be fixed before this file is used further."),
    ("WARNING", WARNING_FILL, "Something unusual or incomplete, but not necessarily wrong. Worth a quick look, not necessarily a blocker."),
    ("ALL GOOD", GOOD_FILL_GUIDE, "No issues of either kind were found for this person."),
]
for label, fill, desc in sev_rows:
    c = ws0.cell(row=r, column=1, value=label)
    c.fill = fill
    c.font = Font(bold=True)
    ws0.merge_cells(start_row=r, start_column=2, end_row=r, end_column=6)
    ws0.cell(row=r, column=2, value=desc).alignment = WRAP
    r += 1
r += 1

ws0.cell(row=r, column=1, value="The 6 error categories").font = SECTION_FONT
ws0.cell(row=r, column=1).fill = SECTION_FILL
ws0.merge_cells(start_row=r, start_column=1, end_row=r, end_column=6)
r += 1

category_explainers = [
    ("File Pairing",
     "Whether every file that was sent out actually came back, and every file that came back can be matched to a real sent file.",
     "If this is broken, the file wasn't checked at all — nothing else in this report can happen for that person until it's fixed."),
    ("Structure",
     "Whether the file's basic shape is intact: the right sheets exist, and every column header is exactly where and what it should be.",
     "Every other check in this report relies on reading columns by their header name. If a header has moved or been overwritten, every check after it becomes unreliable for that sheet."),
    ("Row Identity",
     "Whether every business record (a Customer + Part Number combination) can be reliably matched between the sent file and the returned file.",
     "Rows are matched by WHO they're about, never by which line number they're sitting on — this is what protects the report from being thrown off by an inserted, deleted, or duplicated row."),
    ("Protected Data",
     "Whether anything sales is NOT supposed to touch this cycle — customer details, this year's approved budget, or any column not explicitly allowed — still matches exactly what was sent.",
     "This is the single most important category. A finding here means real, approved data was changed, almost always by accident."),
    ("Calculation",
     "Whether every formula-driven number (Value = Quantity x Price, and the 4 roll-up totals) still actually computes correctly from its own inputs.",
     "A formula can look fine but secretly be broken — turned into a fixed number, showing a #REF! error, or pointing at the wrong cells. This check catches that even when the number on screen looks plausible."),
    ("Pattern/Volume",
     "For columns sales IS allowed to edit, whether the number of changed rows looks statistically normal or unusually high.",
     "Not automatically wrong — just a nudge to double-check that a big change was a genuine bulk update and not an accidental paste over more rows than intended."),
]
for name, what, why in category_explainers:
    ws0.cell(row=r, column=1, value=name).font = Font(bold=True)
    ws0.cell(row=r, column=2, value="What it checks:").font = Font(italic=True)
    ws0.merge_cells(start_row=r, start_column=3, end_row=r, end_column=6)
    ws0.cell(row=r, column=3, value=what).alignment = WRAP
    r += 1
    ws0.cell(row=r, column=2, value="Why it matters:").font = Font(italic=True)
    ws0.merge_cells(start_row=r, start_column=3, end_row=r, end_column=6)
    ws0.cell(row=r, column=3, value=why).alignment = WRAP
    r += 2

ws0.cell(row=r, column=1, value="Full rule reference").font = SECTION_FONT
ws0.cell(row=r, column=1).fill = SECTION_FILL
ws0.merge_cells(start_row=r, start_column=1, end_row=r, end_column=6)
r += 1

ref_header = ["Rule Code", "Category", "What triggers it", "Why it matters", "What to do about it", "Severity"]
for i, h in enumerate(ref_header, start=1):
    cell = ws0.cell(row=r, column=i, value=h)
    cell.font = HEADER_FONT
    cell.fill = HEADER_FILL
r += 1

RULE_REFERENCE = [
    ("INT-002", "File Pairing",
     "A file was sent to someone but never came back, or a returned file doesn't match any known salesperson.",
     "That person's data was never checked at all.",
     "Chase the missing file, or check the filename starts with the salesperson's name correctly.",
     "CRITICAL"),
    ("STR-001", "Structure",
     "A whole sheet (INR / USD / Direct) that was sent is missing from the return file.",
     "That entire sheet's data is unavailable — likely a deleted tab.",
     "Ask the salesperson to resend with the sheet restored.",
     "CRITICAL"),
    ("STR-002/003", "Structure",
     "A column header's text and/or its position doesn't match what was sent (e.g. a label got typed over, or columns shifted).",
     "Every check that reads columns by name becomes unreliable for this sheet from this point on.",
     "Open the file, find the exact column mentioned, and restore the correct header text in the correct position.",
     "CRITICAL"),
    ("STR-004", "Structure",
     "The same column header text appears more than once on one sheet.",
     "It becomes ambiguous which column is the real one — data could be silently misread.",
     "Rename or remove the duplicate header.",
     "CRITICAL"),
    ("STR-005", "Structure",
     "An extra, unexpected sheet was found, or the Customer Code / Part Number columns can't be located at all.",
     "Nothing else can be safely checked on that sheet.",
     "Investigate the file directly — this usually points to a bigger structural problem.",
     "CRITICAL"),
    ("STR-006", "Structure",
     "This person's SENT file (the blank template) doesn't match the layout used for everyone else.",
     "Not sales' fault — but comparisons for this person are being made against a different template than everyone else, which can make other findings for them unreliable.",
     "Fix the master template at the source before this cycle's results are trusted.",
     "CRITICAL"),
    ("ROW-004", "Row Identity",
     "The same Customer Code + Part Number now appears more than once, when it only appeared once in the sent file.",
     "Could be a genuine duplicate order, or an accidental copy-paste.",
     "Check both rows — remove the extra one if it's an accidental duplicate.",
     "WARNING"),
    ("ROW-005", "Row Identity",
     "A row has only ONE of Customer Code / Part Number filled in.",
     "Usually means a row was started and abandoned mid-entry, or a fragment got pasted in by accident.",
     "Fill in the missing field if it's meant to be real, or delete the stray row.",
     "WARNING"),
    ("ROW-007", "Row Identity",
     "A record that existed in the sent file is completely missing from the return file.",
     "Real business data has been lost — someone likely deleted a row by accident.",
     "Restore the row from the original sent file, or confirm with sales why it was removed.",
     "CRITICAL"),
    ("ROW-008", "Row Identity",
     "A record appears in the return file that wasn't in the sent file at all.",
     "Could be legitimate new business, or could be junk/typo data — the finding tells you which looks more likely.",
     "Confirm with sales before including this record in the combined report.",
     "WARNING"),
    ("PRT", "Protected Data",
     "Any value outside this cycle's allowed-edit list — customer details, this year's approved budget, or anything not explicitly permitted — no longer matches what was sent.",
     "This is data corruption, almost always accidental (e.g. the wrong cell was typed into, or a whole column got pasted over).",
     "Always treat as urgent — restore the original value from the sent file.",
     "CRITICAL"),
    ("CAL-001", "Calculation",
     "A number that should be automatically calculated (Value = Quantity x Price, or one of the 4 roll-up totals) doesn't match what its own inputs say it should be.",
     "Usually means a formula broke — turned into a fixed number, shows a #REF! error, or points at the wrong cells. The number can look fine on screen while actually being wrong.",
     "Open the cell in Excel, check whether it's still a real formula, and re-link it to the correct Quantity/Price cells if not.",
     "CRITICAL"),
    ("PAT-001", "Pattern/Volume",
     "A column sales IS allowed to edit (this cycle's forecast, Incoming, EP Price, or Remarks) changed on more than half of all rows.",
     "Not automatically wrong — just statistically unusual, worth a sanity check.",
     "Skim a sample of the changed rows to confirm they look like genuine, deliberate updates.",
     "WARNING"),
]

for code, cat, trigger, why, action, sev in RULE_REFERENCE:
    fill = CRITICAL_FILL if sev == "CRITICAL" else WARNING_FILL
    row_vals = [code, cat, trigger, why, action, sev]
    for i, v in enumerate(row_vals, start=1):
        cell = ws0.cell(row=r, column=i, value=v)
        cell.alignment = WRAP
        if i == 1:
            cell.font = Font(bold=True)
        if i == 6:
            cell.fill = fill
            cell.font = Font(bold=True)
    ws0.row_dimensions[r].height = 45
    r += 1

for col, width in zip("ABCDEF", [14, 16, 34, 34, 34, 12]):
    ws0.column_dimensions[col].width = width
ws0.freeze_panes = "A1"

# ---- Findings sheet ----
ws = wb_report.create_sheet("Findings")
cols = ["Severity", "Rule", "What this means", "Salesperson", "Sheet", "Customer Code",
        "Part Number", "Column", "Old Value", "New Value", "Details"]
for i, c in enumerate(cols, start=1):
    cell = ws.cell(row=1, column=i, value=c)
    cell.font = HEADER_FONT
    cell.fill = HEADER_FILL

sorted_findings = sorted(FINDINGS, key=lambda f: (f["severity"] != "CRITICAL", f["person"], f["sheet"], f["rule"]))
for r, f in enumerate(sorted_findings, start=2):
    row_vals = [
        f["severity"], f["rule"], RULE_DESCRIPTIONS.get(f["rule"], ""), f["person"], f["sheet"],
        f.get("customer_code"), f.get("part_number"), f.get("column"),
        f.get("old_value"), f.get("new_value"), f.get("message"),
    ]
    for c, v in enumerate(row_vals, start=1):
        cell = ws.cell(row=r, column=c, value=v)
        if f["severity"] == "CRITICAL":
            cell.fill = CRITICAL_FILL
        elif f["severity"] == "WARNING":
            cell.fill = WARNING_FILL

widths = [10, 14, 45, 14, 12, 14, 20, 22, 14, 14, 70]
for i, w in enumerate(widths, start=1):
    ws.column_dimensions[get_column_letter(i)].width = w
ws.freeze_panes = "A2"

# ---- Summary sheet ----
ws2 = wb_report.create_sheet("Summary")
ws2.cell(row=1, column=1, value="Salesperson").font = HEADER_FONT
ws2.cell(row=1, column=2, value="Critical").font = HEADER_FONT
ws2.cell(row=1, column=3, value="Warning").font = HEADER_FONT
ws2.cell(row=1, column=4, value="Status").font = HEADER_FONT
for c in range(1, 5):
    ws2.cell(row=1, column=c).fill = HEADER_FILL

by_person = defaultdict(lambda: {"CRITICAL": 0, "WARNING": 0})
for f in FINDINGS:
    by_person[f["person"]][f["severity"]] += 1

row = 2
for person in PAIRS:
    crit, warn = by_person[person]["CRITICAL"], by_person[person]["WARNING"]
    status = "NEEDS REVIEW" if crit else ("REVIEW SUGGESTED" if warn else "ALL GOOD")
    ws2.cell(row=row, column=1, value=person)
    ws2.cell(row=row, column=2, value=crit)
    ws2.cell(row=row, column=3, value=warn)
    ws2.cell(row=row, column=4, value=status)
    row += 1
for i, w in enumerate([16, 10, 10, 18], start=1):
    ws2.column_dimensions[get_column_letter(i)].width = w

# ---- Summary By Category sheet: every person, broken down by rule category ----
CATEGORY_LABELS = {
    "INT": "File Pairing",
    "STR": "Structure",
    "ROW": "Row Identity",
    "PRT": "Protected Data",
    "CAL": "Calculation",
    "PAT": "Pattern/Volume",
}
def rule_category(rule_code):
    prefix = rule_code.split("-")[0]
    return CATEGORY_LABELS.get(prefix, prefix)

ws3 = wb_report.create_sheet("Summary By Category")
cat_cols = list(CATEGORY_LABELS.values())
header_row = ["Salesperson"] + cat_cols + ["Total Critical", "Total Warning", "Overall Status"]
for i, c in enumerate(header_row, start=1):
    cell = ws3.cell(row=1, column=i, value=c)
    cell.font = HEADER_FONT
    cell.fill = HEADER_FILL

GOOD_FILL = PatternFill("solid", fgColor="C6EFCE")

by_person_category = defaultdict(lambda: defaultdict(int))
for f in FINDINGS:
    by_person_category[f["person"]][rule_category(f["rule"])] += 1

row = 2
for person in PAIRS:
    crit, warn = by_person[person]["CRITICAL"], by_person[person]["WARNING"]
    status = "NEEDS REVIEW" if crit else ("REVIEW SUGGESTED" if warn else "ALL GOOD")

    ws3.cell(row=row, column=1, value=person)
    for ci, cat_label in enumerate(cat_cols, start=2):
        count = by_person_category[person].get(cat_label, 0)
        cell = ws3.cell(row=row, column=ci, value=count if count else None)
        if count:
            cell.fill = CRITICAL_FILL if crit else WARNING_FILL

    ws3.cell(row=row, column=len(cat_cols)+2, value=crit)
    ws3.cell(row=row, column=len(cat_cols)+3, value=warn)
    status_cell = ws3.cell(row=row, column=len(cat_cols)+4, value=status)
    status_cell.fill = CRITICAL_FILL if crit else (WARNING_FILL if warn else GOOD_FILL)
    row += 1

ws3.column_dimensions["A"].width = 16
for i in range(2, len(cat_cols)+2):
    ws3.column_dimensions[get_column_letter(i)].width = 15
ws3.column_dimensions[get_column_letter(len(cat_cols)+2)].width = 14
ws3.column_dimensions[get_column_letter(len(cat_cols)+3)].width = 14
ws3.column_dimensions[get_column_letter(len(cat_cols)+4)].width = 18
ws3.freeze_panes = "B2"

wb_report.save(REPORT_FILE)
print("Saved:", REPORT_FILE)


Saved: validation_reports/validation_report_20260830_131636.xlsx


## 9. Run summary — read this first

A plain-English readout of what this run found, person by person. If
someone shows **NEEDS REVIEW**, look at the "Findings" tab in the Excel
report for the exact rows and values involved before combining their file
into the master report.


In [10]:
print("=" * 72)
print("SALES FILE VALIDATION — RUN SUMMARY")
print("=" * 72)
print(f"Run finished : {datetime.now().strftime('%d-%b-%Y %H:%M:%S')}")
print(f"Report file  : {REPORT_FILE}")
print(f"Files checked: {len(PAIRS)} of {len(sent_by_person)} sent, {len(recv_by_person)} received")
print("-" * 72)

overall_critical = 0
for person in sorted(by_person.keys() | set(PAIRS.keys())):
    crit = by_person[person]["CRITICAL"]
    warn = by_person[person]["WARNING"]
    overall_critical += crit

    if crit == 0 and warn == 0:
        print(f"{person:<12} ALL GOOD — nothing to review.")
        continue

    label = "NEEDS REVIEW" if crit else "REVIEW SUGGESTED"
    print(f"{person:<12} {label}  ({crit} critical, {warn} warning)")

    person_findings = [f for f in FINDINGS if f["person"] == person]
    by_rule = Counter(f["rule"] for f in person_findings)
    for rule, count in by_rule.most_common():
        print(f"    - {RULE_DESCRIPTIONS.get(rule, rule)} [{rule}]: {count} row(s)")
    print()

print("-" * 72)
if overall_critical == 0:
    print("RESULT: ALL GOOD — every file passed the critical checks.")
    print("        Check the Excel report for any warnings worth a quick look.")
else:
    print(f"RESULT: NEEDS REVIEW — {overall_critical} critical issue(s) found across the files above.")
    print("        Open the 'Findings' tab in the Excel report for the exact rows and values")
    print("        before including these files in the combined report.")
print("=" * 72)


SALES FILE VALIDATION — RUN SUMMARY
Run finished : 30-Aug-2026 13:16:46
Report file  : validation_reports/validation_report_20260830_131636.xlsx
Files checked: 6 of 6 sent, 6 received
------------------------------------------------------------------------
Abhishekh    NEEDS REVIEW  (2 critical, 0 warning)
    - A formula's result doesn't match what its inputs say it should be [CAL-001]: 2 row(s)

Arpit        NEEDS REVIEW  (2 critical, 1 warning)
    - A formula's result doesn't match what its inputs say it should be [CAL-001]: 2 row(s)
    - An incomplete row (Customer Code or Part Number is missing) [ROW-005]: 1 row(s)

Harsh        NEEDS REVIEW  (1 critical, 0 warning)
    - A column header has moved, changed, or is missing [STR-002/003]: 1 row(s)

Kuldeep      REVIEW SUGGESTED  (0 critical, 2 warning)
    - An allowed column changed on an unusually large share of rows [PAT-001]: 1 row(s)
    - An incomplete row (Customer Code or Part Number is missing) [ROW-005]: 1 row(s)

Mahesh 

In [ ]:



xl1 == load
xl2 == load


xl1['ssss'] = xl['ssddd']

export xl1/export()
